# 03 — Feature Engineering

Build the same pair features for training and test data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
from rapidfuzz import fuzz
ROOT=Path.cwd().resolve()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from preprocessing import prepare_source
TRAIN=ROOT/'data'/'train'
s1=prepare_source(pd.read_csv(TRAIN/'train_source1.tsv',sep='\t'))
s2=prepare_source(pd.read_csv(TRAIN/'train_source2.tsv',sep='\t'))
s3=prepare_source(pd.read_csv(TRAIN/'train_source3.tsv',sep='\t'))
candidates=pd.read_pickle(ROOT/'output'/'candidate_pairs_internal.pkl')
gt=pd.read_csv(TRAIN/'train_ground_truth.tsv',sep='\t')

In [ ]:
s2_map=s2.set_index('entity_id').to_dict('index'); s3_map=s3.set_index('entity_id').to_dict('index'); s1_map=s1.set_index('entity_id').to_dict('index')

def pair_features(row):
    a=s1_map[row.source1_entity_id]
    b=(s2_map|s3_map)[row.candidate_entity_id]
    n1,n2=a['name_norm'],b['name_norm']; ad1,ad2=a['address_norm'],b['address_norm']
    return {
      'name_similarity':fuzz.WRatio(n1,n2)/100,
      'name_token_similarity':fuzz.token_set_ratio(n1,n2)/100,
      'address_similarity':fuzz.WRatio(ad1,ad2)/100,
      'address_token_similarity':fuzz.token_set_ratio(ad1,ad2)/100,
      'name_exact':int(bool(n1) and n1==n2),
      'country_match':int(a['country_norm']==b['country_norm']),
      'name_missing':int(not n1 or not n2),
      'address_missing':int(not ad1 or not ad2),
      'name_length_diff':abs(len(n1)-len(n2)),
      'address_length_diff':abs(len(ad1)-len(ad2))
    }
features=candidates.copy()
extra=pd.DataFrame([pair_features(r) for r in candidates.itertuples(index=False)])
features=pd.concat([features.reset_index(drop=True),extra],axis=1)

In [ ]:
truth={}
for r in gt.itertuples(index=False):
    mids=set(str(r.matched_entity_ids).split(',')) if pd.notna(r.matched_entity_ids) and str(r.matched_entity_ids).strip() else set()
    truth[r.source1_entity_id]=mids
features['label']=[int(r.candidate_entity_id in truth.get(r.source1_entity_id,set())) for r in features.itertuples(index=False)]
print(features['label'].value_counts())

In [ ]:
features.to_pickle(ROOT/'output'/'training_features.pkl')
features.to_csv(ROOT/'output'/'entity_resolution_features_train.tsv',sep='\t',index=False)
print('Saved training features.')